In [55]:
import json
import pandas as pd
from collections import Counter

# Load JSONL into pandas
records = []
with open("data/all_providers_results_parallelized.jsonl", "r") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("Bad line:", e)
df = pd.DataFrame(records)

print(f"✅ Loaded {len(df)} rows")
print(df.columns.tolist())


✅ Loaded 2294 rows
['index', 'question', 'choices', 'correct_answer', 'gemini-2.5-flash-lite_response', 'meta-llama_Llama-3.2-11B-Vision-Instruct_response', 'google_gemma-3n-E4B-it_response', 'openai_gpt-oss-20b_response', 'nvidia_NVIDIA-Nemotron-Nano-9B-v2_response', 'accounts_fireworks_models_kimi-k2-thinking_response', 'openai_gpt-oss-120b_response', 'gpt-5-nano-2025-08-07_response', 'grok-4-fast-reasoning_response']


In [56]:
# Count occurrences of each model key
model_columns = [c for c in df.columns if c.endswith('_response')]
print(f"Models detected ({len(model_columns)} total):")
for m in model_columns:
    print("-", m)



Models detected (9 total):
- gemini-2.5-flash-lite_response
- meta-llama_Llama-3.2-11B-Vision-Instruct_response
- google_gemma-3n-E4B-it_response
- openai_gpt-oss-20b_response
- nvidia_NVIDIA-Nemotron-Nano-9B-v2_response
- accounts_fireworks_models_kimi-k2-thinking_response
- openai_gpt-oss-120b_response
- gpt-5-nano-2025-08-07_response
- grok-4-fast-reasoning_response


In [57]:
counts = {m: df[m].notna().sum() for m in model_columns}
for m, c in counts.items():
    print(f"{m}: {c}")



gemini-2.5-flash-lite_response: 400
meta-llama_Llama-3.2-11B-Vision-Instruct_response: 400
google_gemma-3n-E4B-it_response: 400
openai_gpt-oss-20b_response: 400
nvidia_NVIDIA-Nemotron-Nano-9B-v2_response: 400
accounts_fireworks_models_kimi-k2-thinking_response: 400
openai_gpt-oss-120b_response: 400
gpt-5-nano-2025-08-07_response: 400
grok-4-fast-reasoning_response: 400


In [58]:
import pandas as pd
from IPython.display import display

# Make sure all expected columns exist
for m in model_columns:
    if m not in df.columns:
        df[m] = None

# 1. Group by question index
grouped = df.groupby("index")

# 2. Identify incomplete indices (missing any model response)
incomplete_indices = []
for idx, group in grouped:
    found_models = [m for m in model_columns if group[m].notna().any()]
    if len(found_models) < len(model_columns):
        incomplete_indices.append(idx)

print(f"Total incomplete indices: {len(incomplete_indices)}")
print("First few:", incomplete_indices[:20])

# 3. Gracefully handle empty case
if not incomplete_indices:
    print("\n✅ All indices appear complete after merging.")
else:
    # Pick one incomplete index to inspect
    idx = incomplete_indices[0]
    group = grouped.get_group(idx)
    print(f"\n--- Inspecting index {idx} ---")

    # 4. Merge all fragments (later rows overwrite earlier)
    merged = group.iloc[0].copy()
    for m in model_columns:
        vals = group[m].dropna().tolist()
        if vals:
            merged[m] = vals[-1]  # take most recent non-null

    # 5. Display all responses neatly
    cols_to_show = ["index", "question", "correct_answer"] + model_columns
    pd.set_option("display.max_colwidth", 200)
    display(merged[cols_to_show].to_frame().T)

Total incomplete indices: 0
First few: []

✅ All indices appear complete after merging.


In [59]:
import json
import pandas as pd
from IPython.display import display, HTML

# Load JSONL
records = []
with open("data/all_providers_results_parallelized.jsonl") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass

df = pd.DataFrame(records)

# Group by question index
grouped = df.groupby("index")

def show_group(idx, group):
    """Render one question group neatly."""
    model_cols = [c for c in group.columns if c.endswith("_response")]
    q = group.iloc[0]  # question text and choices (same within group)

    html = f"""
    <h3>Question index: {idx}</h3>
    <p><b>Question:</b> {q['question']}</p>
    <p><b>Choices:</b> {q['choices']}</p>
    <p><b>Correct answer:</b> {q['correct_answer']}</p>
    """

    rows = []
    for _, row in group.iterrows():
        for m in model_cols:
            if m in row and pd.notna(row[m]):
                val = row[m]
                if isinstance(val, dict):
                    answer = val.get("answer", str(val))
                    input_tokens = val.get("input_tokens", "N/A")
                    output_tokens = val.get("output_tokens", "N/A")
                    first_token_latency = val.get("first_token_latency", "N/A")
                    total_latency = val.get("total_latency", "N/A")
                    
                    # Format latency values
                    if isinstance(first_token_latency, (int, float)):
                        first_token_latency = f"{first_token_latency:.3f}s"
                    if isinstance(total_latency, (int, float)):
                        total_latency = f"{total_latency:.3f}s"
                    elif total_latency is None:
                        total_latency = "N/A"
                else:
                    answer = str(val)
                    input_tokens = "N/A"
                    output_tokens = "N/A"
                    first_token_latency = "N/A"
                    total_latency = "N/A"
                
                rows.append({
                    "Model": m.replace("_response", ""),
                    "Answer": answer,
                    "Input Tokens": input_tokens,
                    "Output Tokens": output_tokens,
                    "First Token Latency": first_token_latency,
                    "Total Latency": total_latency
                })
    table = pd.DataFrame(rows)

    html += table.to_html(index=False, escape=False)
    html += "<hr>"
    display(HTML(html))

# Display first 10 unique indices (grouped)
for i, (idx, group) in enumerate(grouped):
    show_group(idx, group)
    if i == 9:
        break


Model,Answer,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Explanation/Reasoning: The passage states that ""Ten percent of women who were depressed had depressed partners."" This matches option 0. It also mentions, ""For the healthy women, the figure was only 2.6 percent."" This contradicts option 1, as it suggests 2.6% of *healthy women* had depressed partners, not that 2.6% of healthy women *were* depressed. The passage clearly indicates that special attention is needed for families with two depressed parents (option 2) and that primary school children with two troubled parents have difficulty relating to their peers (option 3). Therefore, option 1 is the statement that is NOT true based on the provided text.\nFinal Answer: 1",569,149,0.365s,0.756s
meta-llama_Llama-3.2-11B-Vision-Instruct,"Let's take a closer look at the options. Statement 0 says that 10% of women who were depressed had depressed partners, which is actually supported by the passage. It mentions that ""Ten percent of women who were depressed had depressed partners."" So, this statement is true.\n\nStatement 1 claims that 2.6% of healthy women were depressed, but the passage actually says that only 2.6% of healthy women had depressed partners, not that they themselves were depressed. This is a subtle but important distinction.\n\nStatement 2 suggests that special attention should be paid to families in which both the father and the mother were depressed, which is indeed a conclusion drawn from the passage. The researcher in Atlanta mentions that primary school children with two troubled parents have difficulty relating to their peers, so this statement is also true.\n\nStatement 3 says that primary school children whose parents were both depressed couldn't get along well with their peers, which is actually supported by the passage. The researcher in Atlanta mentions that primary school children with two troubled parents have difficulty relating to their peers, so this statement is true.\n\nBased on this analysis, the statement that is NOT true is statement 1, which misrepresents the information in the passage.\n\nFinal Answer: 1",571,255,0.491s,8.706s
google_gemma-3n-E4B-it,"Okay, let's break down this question. The question asks which statement is *not* true according to the passage. I'll go through each option and see if it's supported by the text.\n\n* **0: Ten percent of women who were depressed had depressed partners.** The passage explicitly states, ""Ten percent of women who were depressed had depressed partners."" So, this *is* true.\n\n* **1: 2.6 percent of healthy women were depressed.** The passage says, ""For the healthy women, the figure was only 2.6 percent."" This *is* also true.\n\n* **2: Special attention should be paid to families in which both the father and the mother were depressed.** The passage mentions, ""Previous research suggests that families with two depressed parents may need special attention."" This *is* true.\n\n* **3: Primary school children whose parents were both depressed couldn't get along well with their peers.** The passage states, ""A researcher in Atlanta has found that primary school children with two troubled parents have difficulty relating to their peers."" This *is* true.\n\nHmm, it seems like there's a bit of a trick here! All the statements appear to be true based on the text. Let me re-read carefully to make absolutely sure I haven't missed anything. Okay, I’ve reviewed it again, and all the statements *are* directly supported by the passage. However, the question asks which statement is *NOT* true. This implies there's a subtle nuance or a statement that's not quite as directly supported as the others, or perhaps there's a slight misreading. Let's consider the phrasing of statement 3. It says ""couldn't get along well with their peers"". The passage says they ""have difficulty relating to their peers"". ""Couldn't"" is more absolute than ""have difficulty"". This makes stat

Model,Answer,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"The passage strongly suggests that Kauai is for people who appreciate nature, the outdoors, and a more natural, less developed experience. It explicitly states, ""It's about going natural and finding the nature within you."" It also highlights activities like walking along the coast, diving, swimming, and enjoying the scenery. Conversely, it points out the lack of city life amenities like discos, clubs, and four-star restaurants, and downplays shopping opportunities. Therefore, those who love nature would be most drawn to Kauai.\n\nFinal Answer: 0",690,109,0.277s,0.677s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage heavily criticizes Kauai, focusing on its unpredictable weather, lack of luxury amenities like fine dining and nightlife, limited shopping options, and slow, winding roads. It explicitly contrasts Kauai with the comfort of a fine hotel and swimming pool. The author suggests that Kauai isn't for those seeking pampering or a convenient, predictable vacation. Therefore, the choice that best reflects the type of person the passage *doesn't* recommend visit Kauai is those who love the comfort in a fine hotel.\n\nFinal Answer: 2",698,112,0.761s,4.499s
meta-llama_Llama-3.2-11B-Vision-Instruct,"Based on the passage, it seems that Kauai is not for everyone. The author suggests that it's not a good fit for those who love city life, fine hotels, or shopping. The island is described as a do-it-yourself place that's all about nature and simplicity, with limited amenities and activities.\n\nThe passage specifically states that Kauai is not about pampering, and that it's not a good place for those who want to indulge in luxury or urban entertainment. The author also mentions that the island lacks fine dining options, discos, and clubs, which are typical attractions for those who love city life.\n\nOn the other hand, the passage suggests that Kauai is perfect for those who love nature, the outdoors, and a more laid-back, rustic lifestyle. The author describes the island's scenery as breathtaking, with waterfalls, rivers, and taro fields, and mentions that it's a great place for activities like walking, diving, and swimming.\n\nOverall, it seems that the author is trying to deter people from visiting Kauai if they're not interested in nature and the outdoors, and is instead recommending it for those who are looking for a more natural, peaceful experience.\n\nFinal Answer: 0",709,250,0.341s,8.903s
openai_gpt-oss-20b,"Explanation/Reasoning: The passage repeatedly emphasizes Kauai as a place where you’re encouraged to immerse yourself in natural experiences—walking along the coast, diving, swimming, enjoying beautiful scenery, and experiencing a more “do-it‑yourself” lifestyle. It makes it clear that the island is not about hotels, clubs, or high‑end dining but about nature and self‑discovery. Therefore, those who love nature would be the ones most drawn to visiting Kauai. \nFinal Answer: 0",690,318,0.333s,10.460s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: The passage emphasizes Kauai's natural beauty, outdoor activities, and connection to nature, while criticizing its lack of modern amenities like hotels, shopping, and city life. It explicitly states that Kauai is for those who want to ""go natural"" and enjoy activities like diving, swimming, and exploring nature. The author contrasts this with preferences for comfort, shopping, or city life, which are not aligned with Kauai's offerings. Thus, the target audience is clearly nature enthusiasts. \nFinal Answer: 0",693,450,0.498s,15.518s
openai_gpt-oss-120b,"Explanation/Reasoning: The passage repeatedly emphasizes that Kauai is best suited for people who appreciate natural scenery, outdoor activities, and a DIY, nature‑focused experience. It specifically dismisses fine‑hotel comfort, city life, and shopping as reasons not to visit. Therefore, the group most likely

Model,Answer,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"Well, Toby's really bummed about his Super Racer being stolen, and he's hoping to replace it. He mentions that the toy company only has red, blue, and green ones. But then, his dad found out that Cobb's Toy Store *might* have purple ones. Toby specifically asks his Gramps if they can go there when he visits. So, it's pretty clear he's hoping for a purple one to replace his stolen racer.\n\nFinal Answer: 3",359,102,0.357s,0.820s
meta-llama_Llama-3.2-11B-Vision-Instruct,"Explanation/Reasoning: The writer wants a purple Super Racer, as mentioned in the letter where it's stated that the toy company only has red, blue, and green ones, but Cobb's Toy Store might have purple ones.\nFinal Answer: 3",369,53,0.427s,2.830s
google_gemma-3n-E4B-it,Explanation/Reasoning: Toby specifically asks if they can go to Cobb's Toy Store because they *might* have purple racers. This shows that he wants a purple racer.\n\nFinal Answer: 3,367,43,0.691s,2.325s
openai_gpt-oss-20b,"Explanation/Reasoning:\nThe writer mentions that the toy company only sells red, blue, and green racers, but their dad has learned that Cobb’s Toy Store might have purple ones. The writer is specifically asking to go there to buy a racer, implying they want a purple racer. Therefore the correct choice is ""purple"".\n\nFinal Answer: 3",378,197,0.439s,6.772s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: In the letter, Toby mentions that the stolen racer was a Super Racer. When he contacted the toy company, they only had red, blue, and green options. However, Cobb's Toy Store might have purple ones, which is why Toby is asking to visit them next week. The desire for a purple racer is explicitly stated as the solution to replace the stolen one. \nFinal Answer: 3",364,287,0.491s,10.420s
accounts_fireworks_models_kimi-k2-thinking,"The user is asking me to answer a multiple-choice question based on a letter written by someone named Toby.\n\n Let me read the letter carefully:\n - Toby is writing to his Gramps\n - His Super Racer was stolen from the park\n - He called the toy company and they only have red, blue, and green racers\n - His dad called Cobb's Toy Store and found out they MIGHT have purple ones\n - Toby asks: ""Can we go there when I come to visit you next week?""\n - The question asks: ""The writer wants to have a ___ racer.""\n\n So Toby wants a replacement for his stolen Super Racer. He learned that:\n - The toy company doesn't have purple (only red, blue, green)\n - Cobb's Toy Store MIGHT have purple ones\n - He specifically asks to go to Cobb's Toy Store\n\n The key sentence is: ""Dad called Cobb's Toy Store today and found out they might have purple ones. Can we go there when I come to visit you next week?""\n\n This clearly indicates Toby wants a purple racer. He's specifically interested in going to the store that might have purple ones.\n\n The choices are:\n 0: red\n 1: blue\n 2: green\n 3: purple\n\n The answer should be 3.\n\n I need to respond naturally, as in a conversation, then end with the exact format ""Final Answer: "".\n\n Let me craft a natural response:\n ""Based on the letter, Toby mentions that his dad called Cobb's Toy Store because they might have purple Super Racers, and he specifically asks if they can go there. This shows he wants a purple racer to replace his stolen one. The other colors (red, blue, green) were mentioned as available at the toy company, but Toby is specifically interested in the purple ones at Cobb's Toy Store.""\n\n Then I need to add the final answer line.\n\n Let me make sure I'm following all instructions:\n - Respond naturally: Yes, I'm explaining it conversationally\n - Short or detailed: I'm being detailed enough to be clear\n - End with exact phrase ""Final Answer: "": Yes\n - Format exactly like the example: Yes, with ""Explanation/Reasoning:"" before my explanation.\n\n The index for purple is 3.\n\n Final Answ

Model,Answer,Input Tokens,Output Tokens,First Token Latency,Total Latency
gemini-2.5-flash-lite,"This question is asking what Jean Clottes says about people who visit the mountain. Let's look at his quote: ""People think that because the pictures have been there so long they will always continue to be there. But if the damage continues at this rate there will be nothing left in 50 years."" This directly implies that visitors believe the ancient drawings are permanent and will always exist, regardless of their actions. This aligns with the idea that they ""think the drawings will not disappear.""\n\nFinal Answer: 3",909,106,0.383s,0.722s
meta-llama_Llama-3.2-11B-Vision-Instruct,"It seems that people who visit the mountain are not respecting the ancient drawings and are actually causing harm to them. They're stamping on the drawings, wearing away the rock and definition of the artwork, and even cutting off parts to take home as souvenirs. Some visitors are also using long sticks with sharp ends to scratch their own drawings or names in the rocks, and others are rubbing the drawings to get a clearer picture. This behavior is not only damaging the drawings but also shows a lack of understanding and appreciation for the historical significance of the site.\n\nFinal Answer: 2",905,117,0.373s,4.457s
google_gemma-3n-E4B-it,"Explanation/Reasoning: The passage states that Jean Clottes says, ""People think that because the pictures have been there so long they will always continue to be there."" This implies that people *do* believe the drawings will not disappear, which directly contradicts choice 3. The passage doesn't indicate he thinks they should be left alone (choice 2), or that people are allowed to paint there (choice 1). He is actually concerned about the damage being caused. Therefore, the most accurate inference based on the text is that he is addressing the misconception that the drawings will persist indefinitely.\n\nFinal Answer: 3",917,128,0.937s,4.933s
nvidia_NVIDIA-Nemotron-Nano-9B-v2,"Explanation/Reasoning: Jean Clottes explicitly states that visitors believe the drawings will remain because they have existed for a long time, saying, ""People think that because the pictures have been there so long they will always continue to be there."" This directly aligns with choice 3, which states that visitors ""think the drawings will not disappear."" The other options are not supported by his statements. \nFinal Answer: 3",902,367,0.753s,12.287s
openai_gpt-oss-20b,"Explanation/Reasoning: \nJean Clottes highlights that visitors assume the ancient drawings will always stay untouched because they have existed for so long. This means they believe the drawings will not disappear, even though they are actually harming them. \nFinal Answer: 3",909,637,0.407s,20.156s
accounts_fireworks_models_kimi-k2-thinking,"The user wants me to answer a multiple-choice question based on the provided text.\n The question is: ""Jean Clottes says that people who visit the mountain ___ .""\n The choices are:\n 0: do not believe the drawings are old.\n 1: believe they are allowed to paint there\n 2: think the drawings should be left alone\n 3: think the drawings will not disappear\n\n I need to find what Jean Clottes says about people who visit the mountain in the text.\n\n Let me scan the text for mentions of Jean Clottes and what he says about visitors.\n\n The text says: ""Jean Clottes is the chairman of the International Committee on Rock Art. He says, 'People think that because the pictures have been there so long they will always continue to be there. But if the damage continues at this rate there will be nothing left in 50 years.'""\n\n This quote directly addresses what people (presumably visitors) think. He says they think the pictures have been there so long they will always continue to be there. This matches choice 3: ""think the drawings will not disappear.""\n\n Let me check if there are other statements from Jean Clottes about visitors:\n ""

In [60]:
import pandas as pd
import numpy as np
from IPython.display import display

# Make sure all model columns exist
for m in model_columns:
    if m not in df.columns:
        df[m] = np.nan

# Helper to detect errors in strings or dicts
def is_error(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        val_str = value.strip()
        # Check for "Error:" prefix or error-like patterns
        if val_str.startswith("Error:") or "Error" in val_str or "ConnectionResetError" in val_str:
            return True
    elif isinstance(value, dict):
        # Check if dict has an error field or error in answer
        if "error" in value or (isinstance(value.get("answer"), str) and "Error" in value.get("answer", "")):
            return True
    return False

# Group by index and merge responses from different rows
grouped = df.groupby("index")
issues = []

for idx, group in grouped:
    # Merge all model responses for this index (take last non-null value)
    merged_row = group.iloc[0].copy()
    for m in model_columns:
        vals = group[m].dropna().tolist()
        if vals:
            merged_row[m] = vals[-1]  # take most recent non-null
    
    # Check each model for missing/error responses
    question_text = merged_row.get("question", "")[:120] + "..." if len(merged_row.get("question", "")) > 120 else merged_row.get("question", "")
    
    for m in model_columns:
        val = merged_row.get(m, None)
        if pd.isna(val) or is_error(val):
            issues.append({
                "index": idx,
                "model": m,
                "issue_type": "missing" if pd.isna(val) else "error",
                "question": question_text,
                "value": str(val)[:100] + "..." if val and len(str(val)) > 100 else str(val) if val else None
            })

issues_df = pd.DataFrame(issues)

# --- summary ---
if issues_df.empty:
    print("✅ No missing or error responses found.")
else:
    print(f"⚠️ Found {len(issues_df)} issues across {issues_df['index'].nunique()} questions.")
    print("\n" + "="*60)
    print("Issue type breakdown:")
    print("="*60)
    print(issues_df["issue_type"].value_counts())
    
    # Group errors by model
    print("\n" + "="*60)
    print("Issues grouped by Model:")
    print("="*60)
    model_summary = issues_df.groupby("model").agg({
        "index": "count",
        "issue_type": lambda x: dict(x.value_counts())
    }).rename(columns={"index": "total_count", "issue_type": "breakdown"})
    model_summary = model_summary.sort_values("total_count", ascending=False)
    display(model_summary)
    
    # Group errors by error value/message (for actual errors, not missing)
    error_df = issues_df[issues_df["issue_type"] == "error"].copy()
    if not error_df.empty:
        print("\n" + "="*60)
        print("Errors grouped by Error Message:")
        print("="*60)
        # Extract error message pattern (first part before colon or first 50 chars)
        def extract_error_pattern(x):
            if not x:
                return "Unknown"
            x_str = str(x)
            if ":" in x_str:
                return x_str.split(":")[0].strip()
            return x_str[:50].strip()
        
        error_df["error_pattern"] = error_df["value"].apply(extract_error_pattern)
        error_grouped = error_df.groupby("error_pattern").agg({
            "index": ["count", lambda x: sorted(list(x))[:5]],
            "model": lambda x: sorted(list(x.unique()))[:5]
        })
        error_grouped.columns = ["count", "sample_indices", "sample_models"]
        error_grouped = error_grouped.sort_values("count", ascending=False)
        display(error_grouped)
    
    # Group by model and issue_type (cross-tabulation)
    print("\n" + "="*60)
    print("Issues grouped by Model and Issue Type (Cross-tabulation):")
    print("="*60)
    model_issue_crosstab = pd.crosstab(issues_df["model"], issues_df["issue_type"], margins=True)
    display(model_issue_crosstab)
    
    print("\n" + "="*60)
    print("First few problematic rows:")
    print("="*60)
    display(issues_df.head(20))


⚠️ Found 1 issues across 1 questions.

Issue type breakdown:
issue_type
error    1
Name: count, dtype: int64

Issues grouped by Model:


,total_count,breakdown
model,,
gpt-5-nano-2025-08-07_response,1,{'error': 1}



Errors grouped by Error Message:


,count,sample_indices,sample_models
error_pattern,,,
Error,1,[236],[gpt-5-nano-2025-08-07_response]



Issues grouped by Model and Issue Type (Cross-tabulation):


issue_type,error,All
model,,
gpt-5-nano-2025-08-07_response,1,1
All,1,1



First few problematic rows:


,index,model,issue_type,question,value
0,236,gpt-5-nano-2025-08-07_response,error,They think they're lucky that they're living and it's Christmas again. They can't see that we live on a dirty street in ...,Error: peer closed connection without sending complete message body (incomplete chunked read)
